In [0]:
# NB_patient_360 — Gold Layer: Patient 360 View
# STTM: One row per patient
# Load Strategy: FULL — CREATE OR REPLACE TABLE ... AS SELECT
# Sources: silver.patients (driving), silver.hospitals, silver.insurance_providers, silver.lab_results

from pyspark.sql import functions as F
from pyspark.sql.types import *
import uuid
from datetime import datetime

# --- Configuration ---
CATALOG = "caresync_databricks_workspace"
SOURCE_SCHEMA = "silver"
TARGET_SCHEMA = "gold"
TARGET_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.patient_360"

RUN_ID = str(uuid.uuid4())
RUN_START = datetime.utcnow()

print("=" * 60)
print("NB_patient_360 — Gold Layer Pipeline")
print("=" * 60)
print(f"  Run ID             : {RUN_ID}")
print(f"  Run Start (UTC)    : {RUN_START}")
print(f"  Target Table       : {TARGET_TABLE}")
print("=" * 60)

In [0]:
print("=" * 60)
print("STEP 1: Read Source Tables")
print("=" * 60)

patients_df = spark.table(f"{CATALOG}.{SOURCE_SCHEMA}.patients")
hospitals_df = spark.table(f"{CATALOG}.{SOURCE_SCHEMA}.hospitals")
insurance_df = spark.table(f"{CATALOG}.{SOURCE_SCHEMA}.insurance_providers")
lab_results_df = spark.table(f"{CATALOG}.{SOURCE_SCHEMA}.lab_results")

patients_count = patients_df.count()
hospitals_count = hospitals_df.count()
insurance_count = insurance_df.count()
lab_results_count = lab_results_df.count()

print(f"  patients             : {patients_count:,} rows")
print(f"  hospitals            : {hospitals_count:,} rows")
print(f"  insurance_providers  : {insurance_count:,} rows")
print(f"  lab_results          : {lab_results_count:,} rows")
print("=" * 60)

In [0]:
# Pre-aggregate lab results BEFORE joining to patients to prevent row multiplication.
# Grain: one row per patient_id.
# Patients with no lab results will get NULLs (handled via COALESCE in Step 3).

print("=" * 60)
print("STEP 2: Pre-aggregate Lab Results per Patient")
print("=" * 60)

lab_agg_df = (
    lab_results_df
    .groupBy("patient_id")
    .agg(
        F.count("*").alias("total_tests"),
        F.count(F.when(F.col("result_status") == "Abnormal", 1)).alias("abnormal_test_count"),
        F.count(F.when(F.col("result_status") == "Critical", 1)).alias("critical_test_count"),
        F.max("test_date").alias("last_test_date")
    )
)

lab_agg_count = lab_agg_df.count()
print(f"  Distinct patients with labs : {lab_agg_count:,}")
print(f"  Expected (patients total)   : {patients_count:,}")
print(f"  Patients without labs        : {patients_count - lab_agg_count:,}")
print("=" * 60)

In [0]:
# Join order:
#   patients (driving) 
#   LEFT JOIN hospitals on registered_hospital_id = hospital_id
#   LEFT JOIN insurance_providers on insurance_provider_id
#   LEFT JOIN lab_agg on patient_id
#
# Age calculation: STTM formula — datediff(current_date(), date_of_birth) / 365.25, floored to int
# Note: This is an approximation; may differ by ±1 day vs birthday-aware calculation.
# date_of_birth is timestamp in Silver, cast to date for Gold target.

print("=" * 60)
print("STEP 3: Build Patient 360 Transformation")
print("=" * 60)

patient_360_df = (
    patients_df.alias("p")
    .join(
        hospitals_df.alias("h"),
        F.col("p.registered_hospital_id") == F.col("h.hospital_id"),
        "left"
    )
    .join(
        insurance_df.alias("ip"),
        F.col("p.insurance_provider_id") == F.col("ip.insurance_provider_id"),
        "left"
    )
    .join(
        lab_agg_df.alias("la"),
        F.col("p.patient_id") == F.col("la.patient_id"),
        "left"
    )
    .select(
        F.col("p.patient_id").cast("int").alias("patient_id"),
        F.col("p.first_name").cast("string").alias("first_name"),
        F.col("p.last_name").cast("string").alias("last_name"),
        F.col("p.date_of_birth").cast("date").alias("date_of_birth"),
        # Age: STTM formula — FLOOR(DATEDIFF(current_date, dob) / 365.25)
        F.floor(
            F.datediff(F.current_date(), F.col("p.date_of_birth").cast("date")) / 365.25
        ).cast("int").alias("age"),
        F.col("p.gender").cast("string").alias("gender"),
        F.col("p.city").cast("string").alias("city"),
        F.col("p.state").cast("string").alias("state"),
        F.col("p.registered_hospital_id").cast("int").alias("hospital_id"),
        F.col("h.hospital_name").cast("string").alias("hospital_name"),
        F.col("p.insurance_provider_id").cast("int").alias("insurance_provider_id"),
        F.col("ip.provider_name").cast("string").alias("insurance_provider_name"),
        # Lab aggregates: COALESCE to 0 for patients with no lab results
        F.coalesce(F.col("la.total_tests"), F.lit(0)).cast("int").alias("total_tests"),
        F.coalesce(F.col("la.abnormal_test_count"), F.lit(0)).cast("int").alias("abnormal_test_count"),
        F.coalesce(F.col("la.critical_test_count"), F.lit(0)).cast("int").alias("critical_test_count"),
        F.col("la.last_test_date").cast("date").alias("last_test_date")
    )
)

print(f"  Transformation built successfully.")
print(f"  Output columns: {patient_360_df.columns}")
print("=" * 60)

In [0]:
print("=" * 60)
print("STEP 4: Pre-write Validation")
print("=" * 60)

# Materialize to temp view for validation
patient_360_df.createOrReplaceTempView("patient_360_staging")
staging_count = spark.sql("SELECT COUNT(*) AS cnt FROM patient_360_staging").collect()[0]["cnt"]

# Validation 1: Row count must match patients (1:1 grain)
assert staging_count == patients_count, (
    f"FAIL: Row count {staging_count} != patients count {patients_count}. "
    f"Possible row multiplication or loss from joins."
)
print(f"  [PASS] Row count: {staging_count:,} matches patients ({patients_count:,})")

# Validation 2: Primary key uniqueness
pk_check = spark.sql("""
    SELECT patient_id, COUNT(*) AS cnt 
    FROM patient_360_staging 
    GROUP BY patient_id 
    HAVING cnt > 1
""").count()
assert pk_check == 0, f"FAIL: {pk_check} duplicate patient_id values found."
print(f"  [PASS] patient_id is unique (0 duplicates)")

# Validation 3: No null primary keys
null_pk = spark.sql("SELECT COUNT(*) AS cnt FROM patient_360_staging WHERE patient_id IS NULL").collect()[0]["cnt"]
assert null_pk == 0, f"FAIL: {null_pk} null patient_id values found."
print(f"  [PASS] No null patient_id values")

# Validation 4: total_tests >= 0 (no negatives after COALESCE)
neg_tests = spark.sql("SELECT COUNT(*) AS cnt FROM patient_360_staging WHERE total_tests < 0").collect()[0]["cnt"]
assert neg_tests == 0, f"FAIL: {neg_tests} rows with negative total_tests."
print(f"  [PASS] All total_tests >= 0")

# Validation 5: abnormal + critical <= total
bad_counts = spark.sql("""
    SELECT COUNT(*) AS cnt FROM patient_360_staging 
    WHERE abnormal_test_count > total_tests OR critical_test_count > total_tests
""").collect()[0]["cnt"]
assert bad_counts == 0, f"FAIL: {bad_counts} rows where abnormal/critical > total."
print(f"  [PASS] abnormal/critical counts <= total_tests")

# Validation 6: Schema check
expected_cols = [
    "patient_id", "first_name", "last_name", "date_of_birth", "age", "gender",
    "city", "state", "hospital_id", "hospital_name", "insurance_provider_id",
    "insurance_provider_name", "total_tests", "abnormal_test_count",
    "critical_test_count", "last_test_date"
]
actual_cols = [f.name for f in spark.sql("SELECT * FROM patient_360_staging LIMIT 0").schema.fields]
assert actual_cols == expected_cols, f"FAIL: Column mismatch.\nExpected: {expected_cols}\nActual:   {actual_cols}"
print(f"  [PASS] Schema matches STTM specification ({len(expected_cols)} columns)")

print("-" * 60)
print(f"  All pre-write validations PASSED.")
print("=" * 60)

In [0]:
print("=" * 60)
print(f"STEP 5: Write to Gold — {TARGET_TABLE}")
print("=" * 60)
print(f"  Strategy: CREATE OR REPLACE TABLE ... AS SELECT")

spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE} AS
    SELECT * FROM patient_360_staging
""")

print(f"  ✅ Table created/replaced successfully.")
print("=" * 60)

In [0]:
print("=" * 60)
print("STEP 6: Post-write Verification")
print("=" * 60)

gold_df = spark.table(TARGET_TABLE)
gold_count = gold_df.count()

print(f"  Gold table row count   : {gold_count:,}")
print(f"  Expected (from staging): {staging_count:,}")
assert gold_count == staging_count, f"FAIL: Gold count {gold_count} != staging count {staging_count}"
print(f"  [PASS] Row counts match.")

print("-" * 60)
print("  Schema:")
gold_df.printSchema()
print("-" * 60)
print("  Sample rows (5):")
gold_df.show(5, truncate=False)
print("=" * 60)

In [0]:
from datetime import datetime
import json

RUN_END = datetime.utcnow()
duration_sec = (RUN_END - RUN_START).total_seconds()

exit_payload = {
    "status": "SUCCESS",
    "run_id": RUN_ID,
    "target_table": TARGET_TABLE,
    "rows_written": gold_count,
    "source_patients_count": patients_count,
    "source_lab_results_count": lab_results_count,
    "run_start_utc": str(RUN_START),
    "run_end_utc": str(RUN_END),
    "duration_seconds": round(duration_sec, 2)
}

print("=" * 60)
print("STEP 7: Notebook Exit")
print("=" * 60)
for k, v in exit_payload.items():
    print(f"  {k:30s}: {v}")
print("=" * 60)

In [0]:
record_count = gold_count

print("=" * 60)
print("STEP 8: Simple Record Count Exit")
print("=" * 60)
print(f"  Records processed  : {record_count:,}")
print(f"  Returning value    : {record_count}")
print("=" * 60)

dbutils.notebook.exit(str(record_count))